<a href="https://colab.research.google.com/github/anshupandey/B7_GAAP_GCP/blob/main/code8_Healthcare_Workflow_Agent_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Healthcare Workflow Agents: Patient Intake & Clinical Triage

## Enterprise Use Case — Building Production-Grade Agent Pipelines

**Program:** Generative AI Architect Program — Batch 7 (Day 5 Lab)  
**By:** Blue Data Consulting | April 2026

---

### What You'll Build

This notebook walks you through building a **Patient Intake & Clinical Triage Agent** — a real-world enterprise healthcare system that:

1. Parses unstructured patient intake forms (symptoms, history, medications)
2. Performs clinical triage classification (Emergency / Urgent / Routine)
3. Checks insurance eligibility via API
4. Routes to the correct department
5. Schedules appointments and notifies patients

We build this **incrementally**:
- **Part 1:** Understand the business case and solution architecture
- **Part 2:** Build each step individually with Gemini
- **Part 3:** Compose into a LangGraph workflow (step by step)
- **Part 4:** Rebuild the entire system using Google ADK
- **Part 5:** Compare approaches and discuss production considerations

> **Note:** This notebook uses simulated APIs and mock data. In production, you'd connect to real EHR systems (Epic/Cerner), insurance payers, and scheduling APIs.


---

# Part 1: Business Case & Solution Architecture

## 1.1 The Problem

A large multi-specialty hospital group (15 locations, 2M+ patient visits/year) faces:

| Problem | Impact |
|---------|--------|
| Manual intake processing | **45 minutes** avg per patient intake |
| Triage errors | **12%** of patients misclassified → delayed care |
| Insurance verification delays | **30%** of visits have eligibility issues discovered at appointment time |
| Scheduling inefficiency | **22%** no-show rate due to poor slot matching |
| Staff burnout | Intake staff turnover at **35%** annually |

## 1.2 The Solution

An AI-powered **Patient Intake & Triage Workflow Agent** that:

- Extracts structured clinical data from intake forms (PDF, images, typed text)
- Classifies triage urgency using clinical guidelines (ESI - Emergency Severity Index)
- Verifies insurance in real-time before scheduling
- Routes to the correct department and schedules optimally
- Maintains HIPAA compliance throughout

## 1.3 Expected Business Impact

| Metric | Before | After | Improvement |
|--------|--------|-------|-------------|
| Intake processing time | 45 min | 8 min | **82% reduction** |
| Triage accuracy | 88% | 96% | **+8 percentage points** |
| Insurance issues at visit | 30% | 5% | **83% reduction** |
| No-show rate | 22% | 12% | **45% reduction** |
| Staff processing cost | $18/intake | $3/intake | **83% cost reduction** |


## 1.4 Technical Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                        CLIENT LAYER                                  │
│   Patient Portal (Web/Mobile)  │  Staff Dashboard  │  EHR Plugin     │
└────────────────────────┬────────────────────────────────────────────┘
                         │ HTTPS / REST API
┌────────────────────────▼────────────────────────────────────────────┐
│                     API GATEWAY (Cloud Run)                          │
│              FastAPI + Auth + Rate Limiting                          │
└────────────────────────┬────────────────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────────────────┐
│              WORKFLOW AGENT (LangGraph / ADK)                        │
│                                                                      │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────────────┐ │
│  │ 1. Parse  │──▶│ 2.Triage │──▶│3.Insure  │──▶│ 4. Route +      │ │
│  │  Intake   │   │ Classify │   │  Check   │   │    Schedule      │ │
│  └──────────┘   └──────────┘   └──────────┘   └──────────────────┘ │
│       │              │              │                │               │
│   Gemini Flash   Gemini Pro    API Tool      Conditional Edge       │
│                                                                      │
└────────────────────────┬────────────────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────────────────┐
│                      DATA & SERVICES                                 │
│  AlloyDB (Patient DB)  │  Vertex AI  │  Insurance APIs  │  EHR API  │
│  Cloud Storage (docs)  │  Vector Search (KB)  │  Pub/Sub (events)   │
└─────────────────────────────────────────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────────────────┐
│                   OBSERVABILITY & SECURITY                           │
│  Cloud Trace  │  Langfuse  │  VPC-SC  │  CMEK  │  Audit Logs       │
└─────────────────────────────────────────────────────────────────────┘
```


---

# Part 2: Setup & Building Individual Components

Let's start by installing dependencies and building each agent step individually before composing them into a workflow.


In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
# Uncomment and run these in your environment:

# !pip install google-genai langchain-google-genai langgraph
# !pip install google-adk pydantic
# !pip install google-cloud-aiplatform

# For this notebook, we also need:
# !pip install rich tabulate

print("✅ Dependencies ready")
print("📋 Required: google-genai, langgraph, google-adk, pydantic")


In [ ]:
# ============================================================
# Cell 2: Configuration & Client Setup
# ============================================================
import os
import json
import time
import uuid
from datetime import datetime, timedelta
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field, validator

# Initialize client
from google import genai
client = genai.Client()

print("✅ Client initialized")


## 2.1 Data Models — Pydantic Schemas

Before we build any agent logic, we define **strict data models** using Pydantic. This ensures:
- Every LLM output is validated before it flows downstream
- Schema violations are caught early (not at the database layer)
- The state contract between workflow nodes is explicit and testable


In [ ]:
# ============================================================
# Cell 3: Pydantic Data Models
# ============================================================

class TriageLevel(str, Enum):
    """Emergency Severity Index (ESI) levels used in US hospitals."""
    EMERGENCY = "EMERGENCY"      # ESI 1: Immediate life-threatening
    URGENT = "URGENT"            # ESI 2: High risk / confused / severe pain
    LESS_URGENT = "LESS_URGENT"  # ESI 3: Needs multiple resources
    ROUTINE = "ROUTINE"          # ESI 4-5: One resource or none needed

class VitalSigns(BaseModel):
    blood_pressure: str = Field(..., description="Systolic/Diastolic e.g. '120/80'")
    heart_rate: int = Field(..., ge=30, le=250, description="BPM")
    temperature: str = Field(..., description="Body temperature with unit")
    respiratory_rate: int = Field(..., ge=5, le=60, description="Breaths per minute")
    oxygen_saturation: str = Field(..., description="SpO2 percentage")

class PatientIntake(BaseModel):
    """Structured output from intake form parsing."""
    patient_name: str = Field(..., min_length=2, max_length=100)
    age: int = Field(..., ge=0, le=120)
    gender: str
    chief_complaint: str = Field(..., min_length=10)
    symptoms: list[str] = Field(..., min_length=1)
    duration: str
    medical_history: list[str] = []
    current_medications: list[str] = []
    allergies: list[str] = []
    vital_signs: Optional[VitalSigns] = None

class TriageResult(BaseModel):
    """Output from the triage classification step."""
    triage_level: TriageLevel
    esi_score: int = Field(..., ge=1, le=5)
    reasoning: str = Field(..., min_length=20)
    confidence: float = Field(..., ge=0.0, le=1.0)
    recommended_department: str
    time_sensitivity: str
    red_flags: list[str] = []

class InsuranceEligibility(BaseModel):
    """Output from insurance verification."""
    eligible: bool
    plan: str
    member_id: str
    copay_er: Optional[float] = None
    deductible_remaining: Optional[float] = None
    prior_auth_required: bool = False
    network_status: str = "Unknown"

class SchedulingResult(BaseModel):
    """Output from the scheduling step."""
    appointment_scheduled: bool
    department: str
    arrival_instruction: str
    estimated_wait: str
    notification_sent: bool = False
    notification_channels: list[str] = []

# Validate that our models compile correctly
sample = PatientIntake(
    patient_name="Test Patient",
    age=45,
    gender="Female",
    chief_complaint="Persistent headache for 3 days with visual disturbances",
    symptoms=["headache", "blurred vision"],
    duration="3 days"
)
print(f"✅ Pydantic models validated successfully")
print(f"   Sample: {sample.patient_name}, Age {sample.age}")
print(f"   Symptoms: {sample.symptoms}")
print(f"\n📋 Models defined: PatientIntake, TriageResult, InsuranceEligibility, SchedulingResult")


## 2.2 Building Individual Components

Now we build each step as an independent, testable function. This is the **decomposition** phase — before we wire them into a workflow.

### Step 1: Parse Intake Form

The first node takes raw patient input (text, form data, or document) and extracts structured clinical data.


In [ ]:
# ============================================================
# Cell 4: Step 1 — Parse Intake Form
# ============================================================

# ── The prompt template ──
INTAKE_PARSE_PROMPT = """You are a medical intake specialist. Parse the following patient
intake information and extract structured clinical data.

IMPORTANT:
- Extract ALL symptoms mentioned, including implied ones
- Normalize medication names to generic + dosage
- Flag any allergies explicitly
- If vital signs are provided, extract them precisely
- Use clinical terminology where appropriate

Patient Intake Text:
{intake_text}

Respond ONLY with valid JSON matching this schema:
{{
    "patient_name": "string",
    "age": integer,
    "gender": "string",
    "chief_complaint": "string (concise clinical summary)",
    "symptoms": ["list of symptoms"],
    "duration": "string",
    "medical_history": ["list of conditions"],
    "current_medications": ["medication + dosage"],
    "allergies": ["list of allergies"],
    "vital_signs": {{
        "blood_pressure": "systolic/diastolic",
        "heart_rate": integer,
        "temperature": "value with unit",
        "respiratory_rate": integer,
        "oxygen_saturation": "percentage"
    }}
}}
"""

def parse_intake(intake_text: str) -> PatientIntake:
    """
    Parse raw intake text into structured PatientIntake using Gemini.

    In production:
        response = client.models.generate_content(
            model="gemini-2.5-flash",    # Flash for speed — intake parsing is straightforward
            contents=INTAKE_PARSE_PROMPT.format(intake_text=intake_text),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=PatientIntake,      # Gemini structured output
                temperature=0.1                      # Low temp for factual extraction
            )
        )
    """
    prompt = INTAKE_PARSE_PROMPT.format(intake_text=intake_text)
    response = client.generate(model="gemini-2.5-flash", prompt=prompt)
    parsed = json.loads(response.text)

    # Validate with Pydantic — catches malformed LLM output
    patient = PatientIntake(**parsed)
    return patient


# ── Test with a realistic intake form ──
sample_intake = """
Patient: Rajesh Kumar, 58 year old male

Presenting Complaint: Patient reports severe chest pain that started approximately
2 hours ago. Pain is substernal, radiating to the left arm. Associated with
shortness of breath, profuse sweating. Pain is described as 8/10, crushing in nature.

Past Medical History: Type 2 Diabetes (diagnosed 2015), Hypertension (2012),
High cholesterol

Current Medications: Metformin 500mg twice daily, Amlodipine 5mg daily,
Atorvastatin 20mg at bedtime

Allergies: Penicillin (rash)

Vitals: BP 165/95, HR 102, Temp 98.6°F, RR 22, SpO2 94% on room air
"""

print("=" * 70)
print("STEP 1: PARSE INTAKE FORM")
print("=" * 70)

patient_data = parse_intake(sample_intake)

print(f"\n👤 Patient: {patient_data.patient_name}, Age {patient_data.age}, {patient_data.gender}")
print(f"🏥 Chief Complaint: {patient_data.chief_complaint}")
print(f"📋 Symptoms: {', '.join(patient_data.symptoms)}")
print(f"💊 Medications: {', '.join(patient_data.current_medications)}")
print(f"⚠️  Allergies: {', '.join(patient_data.allergies)}")
if patient_data.vital_signs:
    vs = patient_data.vital_signs
    print(f"🩺 Vitals: BP {vs.blood_pressure}, HR {vs.heart_rate}, SpO2 {vs.oxygen_saturation}")
print(f"\n✅ Parsed successfully — Pydantic validation passed")


### Step 2: Clinical Triage Classification

This is the **most critical** step — it determines patient urgency. We use **Gemini Pro** (not Flash) here because:
- Clinical reasoning requires deeper analysis
- False negatives (missing emergencies) are life-threatening
- We need detailed reasoning for audit trails

We also implement **confidence gating**: if the model's confidence is below 0.8, we flag for human (nurse) review.


In [ ]:
# ============================================================
# Cell 5: Step 2 — Clinical Triage Classification
# ============================================================

TRIAGE_PROMPT = """You are an experienced emergency triage nurse using the Emergency Severity
Index (ESI) v4 algorithm. Classify the following patient based on their clinical presentation.

ESI Levels:
- ESI 1 (EMERGENCY): Immediate life-saving intervention needed. Examples: cardiac arrest,
  severe respiratory distress, major trauma, active stroke symptoms.
- ESI 2 (URGENT): High-risk situation, confused/lethargic/disoriented, or severe pain/distress.
  Should not wait. Examples: chest pain with cardiac risk factors, acute abdomen, high fever
  with immunocompromised status.
- ESI 3 (LESS_URGENT): Needs 2+ resources (labs, imaging, IV meds, specialist consult) but
  vital signs stable. Examples: abdominal pain needing CT, laceration needing sutures + tetanus.
- ESI 4-5 (ROUTINE): Needs 1 resource (ESI 4) or no resources (ESI 5). Examples: prescription
  refill, simple wound, mild URI.

Patient Data:
{patient_json}

CRITICAL RULES:
1. When in doubt between two levels, ALWAYS choose the MORE URGENT level
2. Chest pain in patients >40 with ANY cardiac risk factor = minimum ESI 2
3. Consider vital sign abnormalities as independent escalation criteria
4. Document ALL red flags explicitly

Respond ONLY with valid JSON:
{{
    "triage_level": "EMERGENCY|URGENT|LESS_URGENT|ROUTINE",
    "esi_score": 1-5,
    "reasoning": "detailed clinical reasoning (minimum 50 words)",
    "confidence": 0.0-1.0,
    "recommended_department": "department name",
    "time_sensitivity": "timeframe description",
    "red_flags": ["list of concerning findings"]
}}
"""

# ── Confidence threshold for human review ──
TRIAGE_CONFIDENCE_THRESHOLD = 0.80

def classify_triage(patient: PatientIntake) -> tuple[TriageResult, bool]:
    """
    Classify patient triage level using Gemini Pro.

    Returns:
        (TriageResult, needs_human_review: bool)

    In production:
        response = client.models.generate_content(
            model="gemini-2.5-pro",      # Pro for clinical reasoning
            contents=TRIAGE_PROMPT.format(patient_json=patient.model_dump_json()),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=TriageResult,
                temperature=0.2           # Low temp but not zero — allows some reasoning flexibility
            )
        )
    """
    prompt = TRIAGE_PROMPT.format(patient_json=patient.model_dump_json(indent=2))
    response = client.generate(model="gemini-2.5-pro", prompt=prompt)
    parsed = json.loads(response.text)

    triage = TriageResult(**parsed)

    # Confidence gating
    needs_review = triage.confidence < TRIAGE_CONFIDENCE_THRESHOLD
    return triage, needs_review


# ── Test ──
print("=" * 70)
print("STEP 2: CLINICAL TRIAGE CLASSIFICATION")
print("=" * 70)

triage_result, needs_review = classify_triage(patient_data)

level_emoji = {
    "EMERGENCY": "🔴",
    "URGENT": "🟠",
    "LESS_URGENT": "🟡",
    "ROUTINE": "🟢"
}

print(f"\n{level_emoji.get(triage_result.triage_level, '⚪')} Triage Level: "
      f"{triage_result.triage_level} (ESI {triage_result.esi_score})")
print(f"📊 Confidence: {triage_result.confidence:.0%}")
print(f"🏥 Department: {triage_result.recommended_department}")
print(f"⏰ Time Sensitivity: {triage_result.time_sensitivity}")
print(f"\n📝 Clinical Reasoning:")
print(f"   {triage_result.reasoning}")
print(f"\n🚩 Red Flags:")
for flag in triage_result.red_flags:
    print(f"   • {flag}")

if needs_review:
    print(f"\n⚠️  HUMAN REVIEW REQUIRED — Confidence {triage_result.confidence:.0%} "
          f"below threshold {TRIAGE_CONFIDENCE_THRESHOLD:.0%}")
else:
    print(f"\n✅ Confidence above threshold — proceeding automatically")


### Step 3: Insurance Eligibility Check

This step calls an external insurance verification API. In production, this connects to payer systems like Availity, Change Healthcare, or direct payer APIs.

Key design decisions:
- This is a **tool call** (API), not an LLM reasoning step
- We implement **retry with exponential backoff** for transient failures
- If the API is unreachable, we **proceed with a warning** (don't block emergency care)


In [ ]:
# ============================================================
# Cell 6: Step 3 — Insurance Eligibility Check
# ============================================================
import random

# ── Simulated Insurance API ──
class InsuranceAPI:
    """
    Simulates an insurance eligibility verification service.

    In production, this would be:
    - Availity API (https://developer.availity.com/)
    - Change Healthcare Eligibility API
    - Direct payer API (UHC, Aetna, BCBS, etc.)
    - CMS Medicare Beneficiary API
    """

    # Simulated database of insurance records
    PLANS = {
        "Rajesh Kumar": {
            "eligible": True,
            "plan": "UnitedHealth Gold PPO",
            "member_id": "UHC-2024-889012",
            "copay_er": 150,
            "copay_urgent_care": 50,
            "copay_specialist": 40,
            "deductible_remaining": 850,
            "out_of_pocket_max_remaining": 3200,
            "prior_auth_required": False,
            "network_status": "In-Network"
        }
    }

    def check_eligibility(self, patient_name: str, simulate_failure: bool = False) -> dict:
        """Check insurance eligibility. May raise ConnectionError."""
        if simulate_failure:
            raise ConnectionError("Insurance API temporarily unavailable")

        record = self.PLANS.get(patient_name)
        if record:
            return record
        return {
            "eligible": False,
            "plan": "Unknown",
            "member_id": "N/A",
            "prior_auth_required": False,
            "network_status": "Unknown"
        }


insurance_api = InsuranceAPI()


def check_insurance(
    patient_name: str,
    max_retries: int = 3,
    base_delay: float = 1.0
) -> InsuranceEligibility:
    """
    Check insurance eligibility with retry logic.

    Implements exponential backoff with jitter for production resilience.
    If all retries fail, returns a 'status unknown' result (don't block care).
    """
    for attempt in range(max_retries):
        try:
            result = insurance_api.check_eligibility(patient_name)
            return InsuranceEligibility(**result)

        except ConnectionError as e:
            wait = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"   ⚠️  Attempt {attempt + 1}/{max_retries} failed: {e}")
            print(f"   ⏳ Retrying in {wait:.1f}s...")
            time.sleep(min(wait, 0.5))  # Capped for notebook demo

    # All retries exhausted — return unknown status
    print(f"   ❌ All {max_retries} attempts failed. Proceeding with unknown insurance status.")
    return InsuranceEligibility(
        eligible=False,
        plan="VERIFICATION_FAILED",
        member_id="N/A",
        prior_auth_required=False,
        network_status="Unable to verify — manual check required"
    )


# ── Test ──
print("=" * 70)
print("STEP 3: INSURANCE ELIGIBILITY CHECK")
print("=" * 70)

insurance = check_insurance(patient_data.patient_name)

print(f"\n💳 Plan: {insurance.plan}")
print(f"🆔 Member ID: {insurance.member_id}")
print(f"✅ Eligible: {insurance.eligible}")
print(f"🏥 Network: {insurance.network_status}")
if insurance.copay_er:
    print(f"💰 ER Copay: ${insurance.copay_er}")
if insurance.deductible_remaining:
    print(f"📉 Deductible Remaining: ${insurance.deductible_remaining}")
print(f"📋 Prior Auth Required: {insurance.prior_auth_required}")


### Step 4: Route & Schedule

Based on triage level, we route the patient to the correct department and handle scheduling. This step uses **conditional logic** (not LLM) for routing, which is more predictable and auditable.


In [ ]:
# ============================================================
# Cell 7: Step 4 — Route & Schedule
# ============================================================

# ── Routing rules (deterministic, not LLM-based) ──
ROUTING_TABLE = {
    TriageLevel.EMERGENCY: {
        "department": "Emergency Department",
        "action": "IMMEDIATE - Direct to ED triage bay",
        "notify": ["ED Attending", "Charge Nurse", "Cardiology On-Call"],
        "estimated_wait": "0 minutes (priority triage)",
    },
    TriageLevel.URGENT: {
        "department": "Urgent Care / Fast Track",
        "action": "Priority queue — seen within 30 minutes",
        "notify": ["Urgent Care Physician"],
        "estimated_wait": "15-30 minutes",
    },
    TriageLevel.LESS_URGENT: {
        "department": "General Clinic",
        "action": "Schedule same-day appointment if available",
        "notify": ["Front Desk"],
        "estimated_wait": "1-2 hours",
    },
    TriageLevel.ROUTINE: {
        "department": "Primary Care",
        "action": "Schedule next available appointment",
        "notify": ["Scheduling Team"],
        "estimated_wait": "Next available slot (1-3 days)",
    },
}

def route_and_schedule(
    triage: TriageResult,
    patient: PatientIntake,
    insurance: InsuranceEligibility
) -> SchedulingResult:
    """
    Route patient to department and schedule based on triage level.

    Uses deterministic routing rules (not LLM) for:
    - Predictability: same triage level always routes the same way
    - Auditability: routing logic is explicit and reviewable
    - Speed: no API call needed for routing decisions
    """
    route = ROUTING_TABLE[triage.triage_level]

    # Log the routing decision
    print(f"   📍 Routing to: {route['department']}")
    print(f"   📋 Action: {route['action']}")
    print(f"   📞 Notifying: {', '.join(route['notify'])}")

    # In production: call EHR scheduling API (Epic FHIR, Cerner, etc.)
    # appointment = ehr_client.schedule(
    #     department=route["department"],
    #     patient_id=patient.patient_name,
    #     urgency=triage.triage_level
    # )

    # Build scheduling result
    result = SchedulingResult(
        appointment_scheduled=True,
        department=route["department"],
        arrival_instruction=route["action"],
        estimated_wait=route["estimated_wait"],
        notification_sent=True,
        notification_channels=["SMS", "Patient Portal"]
    )

    return result


# ── Test ──
print("=" * 70)
print("STEP 4: ROUTE & SCHEDULE")
print("=" * 70)

scheduling = route_and_schedule(triage_result, patient_data, insurance)

print(f"\n🏥 Department: {scheduling.department}")
print(f"📋 Instructions: {scheduling.arrival_instruction}")
print(f"⏱️  Wait: {scheduling.estimated_wait}")
print(f"📱 Notifications: {', '.join(scheduling.notification_channels)}")


---

# Part 3: Composing with LangGraph — Step by Step

Now we take our individual components and wire them into a **stateful directed graph** using LangGraph.

### Why LangGraph?

- **Explicit control flow** — You see every node and edge
- **Typed state** — State dict flows through the graph, validated at each step
- **Conditional edges** — Branch on triage level, insurance status, confidence
- **Checkpointing** — Pause for human review, resume later
- **Observability** — Every node transition is traceable

Let's build it incrementally.


In [ ]:
# ============================================================
# Cell 8: LangGraph — Define the State Schema
# ============================================================

# In production, you'd install and import:
# from langgraph.graph import StateGraph, END
# from langgraph.checkpoint.memory import MemorySaver

# For this notebook, we build a lightweight StateGraph equivalent
# that demonstrates the exact same pattern.

from typing import TypedDict, Any, Callable


class PatientWorkflowState(TypedDict, total=False):
    """
    The shared state that flows through our workflow graph.

    Every node reads from this state and writes back to it.
    Keys are added incrementally as the workflow progresses.
    """
    # Input
    intake_text: str                    # Raw patient intake text
    run_id: str                         # Unique run ID for tracing

    # Step 1 output
    patient_data: dict                  # Parsed PatientIntake (as dict for serialization)
    parse_error: str                    # Error message if parsing failed

    # Step 2 output
    triage_result: dict                 # TriageResult (as dict)
    needs_human_review: bool            # Confidence gating flag
    triage_error: str

    # Step 3 output
    insurance: dict                     # InsuranceEligibility (as dict)
    insurance_error: str

    # Step 4 output
    scheduling: dict                    # SchedulingResult (as dict)
    routing_decision: str               # Which department

    # Final output
    summary: str                        # Human-readable summary
    workflow_status: str                # "completed" | "failed" | "needs_review"
    processing_time_ms: float
    errors: list                        # Accumulated errors


print("✅ PatientWorkflowState schema defined")
print("\nState keys flow through the graph:")
for key in PatientWorkflowState.__annotations__:
    print(f"   • {key}: {PatientWorkflowState.__annotations__[key]}")


In [ ]:
# ============================================================
# Cell 9: LangGraph — Define Node Functions
# ============================================================

def node_parse_intake(state: dict) -> dict:
    """Node 1: Parse the intake text into structured data."""
    print("\n🔄 [Node: parse_intake] Processing...")
    start = time.time()

    try:
        patient = parse_intake(state["intake_text"])
        elapsed = (time.time() - start) * 1000
        print(f"   ✅ Parsed: {patient.patient_name}, {len(patient.symptoms)} symptoms ({elapsed:.0f}ms)")
        return {
            "patient_data": patient.model_dump(),
            "parse_error": None
        }
    except Exception as e:
        print(f"   ❌ Parse error: {e}")
        return {
            "patient_data": None,
            "parse_error": str(e),
            "workflow_status": "failed",
            "errors": state.get("errors", []) + [f"Parse error: {e}"]
        }


def node_triage(state: dict) -> dict:
    """Node 2: Classify triage level."""
    print("\n🔄 [Node: triage] Classifying...")

    if state.get("parse_error"):
        return {"triage_error": "Skipped — upstream parse error"}

    try:
        patient = PatientIntake(**state["patient_data"])
        triage, needs_review = classify_triage(patient)
        print(f"   ✅ Triage: {triage.triage_level} (confidence: {triage.confidence:.0%})")
        return {
            "triage_result": triage.model_dump(),
            "needs_human_review": needs_review,
            "triage_error": None
        }
    except Exception as e:
        print(f"   ❌ Triage error: {e}")
        return {
            "triage_result": None,
            "triage_error": str(e),
            "errors": state.get("errors", []) + [f"Triage error: {e}"]
        }


def node_insurance(state: dict) -> dict:
    """Node 3: Check insurance eligibility."""
    print("\n🔄 [Node: insurance] Checking eligibility...")

    if not state.get("patient_data"):
        return {"insurance_error": "Skipped — no patient data"}

    try:
        patient = PatientIntake(**state["patient_data"])
        ins = check_insurance(patient.patient_name)
        print(f"   ✅ Insurance: {ins.plan} (Eligible: {ins.eligible})")
        return {
            "insurance": ins.model_dump(),
            "insurance_error": None
        }
    except Exception as e:
        print(f"   ❌ Insurance error: {e}")
        return {
            "insurance": InsuranceEligibility(
                eligible=False, plan="ERROR", member_id="N/A",
                prior_auth_required=False, network_status="Error"
            ).model_dump(),
            "insurance_error": str(e),
            "errors": state.get("errors", []) + [f"Insurance error: {e}"]
        }


def node_route_schedule(state: dict) -> dict:
    """Node 4: Route to department and schedule."""
    print("\n🔄 [Node: route_schedule] Routing...")

    if not state.get("triage_result"):
        return {"scheduling": None, "routing_decision": "FAILED"}

    try:
        triage = TriageResult(**state["triage_result"])
        patient = PatientIntake(**state["patient_data"])
        insurance = InsuranceEligibility(**state["insurance"]) if state.get("insurance") else None

        scheduling = route_and_schedule(triage, patient, insurance)
        print(f"   ✅ Routed to: {scheduling.department}")
        return {
            "scheduling": scheduling.model_dump(),
            "routing_decision": triage.recommended_department
        }
    except Exception as e:
        print(f"   ❌ Routing error: {e}")
        return {
            "scheduling": None,
            "routing_decision": "ERROR",
            "errors": state.get("errors", []) + [f"Routing error: {e}"]
        }


def node_format_summary(state: dict) -> dict:
    """Node 5: Generate a human-readable summary."""
    print("\n🔄 [Node: format_summary] Generating summary...")

    patient = state.get("patient_data", {})
    triage = state.get("triage_result", {})
    insurance = state.get("insurance", {})
    scheduling = state.get("scheduling", {})

    summary_parts = []
    summary_parts.append(f"Patient: {patient.get('patient_name', 'Unknown')}, "
                         f"Age {patient.get('age', '?')}")
    summary_parts.append(f"Chief Complaint: {patient.get('chief_complaint', 'N/A')}")

    level = triage.get('triage_level', 'UNKNOWN')
    emoji = {"EMERGENCY": "🔴", "URGENT": "🟠", "LESS_URGENT": "🟡", "ROUTINE": "🟢"}.get(level, "⚪")
    summary_parts.append(f"Triage: {emoji} {level} (ESI {triage.get('esi_score', '?')})")
    summary_parts.append(f"Confidence: {triage.get('confidence', 0):.0%}")
    summary_parts.append(f"Department: {scheduling.get('department', 'N/A')}")
    summary_parts.append(f"Insurance: {insurance.get('plan', 'N/A')} "
                         f"({'Eligible' if insurance.get('eligible') else 'Not Eligible'})")

    if scheduling.get('arrival_instruction'):
        summary_parts.append(f"Instructions: {scheduling['arrival_instruction']}")

    summary = "\n".join(summary_parts)

    status = "completed"
    if state.get("needs_human_review"):
        status = "needs_review"
    if state.get("errors"):
        status = "completed_with_errors"

    print(f"   ✅ Summary generated ({len(summary)} chars)")
    return {
        "summary": summary,
        "workflow_status": status
    }


print("✅ All 5 node functions defined")
print("   1. node_parse_intake  → Gemini Flash")
print("   2. node_triage        → Gemini Pro")
print("   3. node_insurance     → API Tool")
print("   4. node_route_schedule → Deterministic Rules")
print("   5. node_format_summary → Summary Generation")


In [ ]:
# ============================================================
# Cell 10: LangGraph — Build & Run the Workflow
# ============================================================

class SimpleWorkflowGraph:
    """
    Lightweight workflow graph that mirrors LangGraph's API.

    In production, replace with:
        from langgraph.graph import StateGraph, END
        graph = StateGraph(PatientWorkflowState)
        graph.add_node("parse_intake", node_parse_intake)
        graph.add_node("triage", node_triage)
        graph.add_conditional_edges("triage", route_by_confidence, {...})
        ...
        app = graph.compile(checkpointer=MemorySaver())
    """

    def __init__(self):
        self.nodes: dict[str, Callable] = {}
        self.edges: list[tuple[str, str]] = []
        self.conditional_edges: dict[str, tuple[Callable, dict]] = {}
        self.entry_point: str = None

    def add_node(self, name: str, func: Callable):
        self.nodes[name] = func

    def add_edge(self, from_node: str, to_node: str):
        self.edges.append((from_node, to_node))

    def add_conditional_edges(self, from_node: str, condition_func: Callable, mapping: dict):
        self.conditional_edges[from_node] = (condition_func, mapping)

    def set_entry_point(self, name: str):
        self.entry_point = name

    def invoke(self, initial_state: dict) -> dict:
        """Execute the workflow graph."""
        state = {**initial_state, "errors": [], "run_id": str(uuid.uuid4())[:8]}
        start_time = time.time()

        print("=" * 70)
        print(f"🚀 WORKFLOW STARTED — Run ID: {state['run_id']}")
        print("=" * 70)

        # Build adjacency from edges
        next_map = {}
        for src, dst in self.edges:
            next_map[src] = dst

        current = self.entry_point
        visited = []

        while current and current != "END":
            if current not in self.nodes:
                print(f"\n❌ Unknown node: {current}")
                break

            visited.append(current)
            # Execute node
            result = self.nodes[current](state)
            state.update(result)

            # Check conditional edges first
            if current in self.conditional_edges:
                cond_func, mapping = self.conditional_edges[current]
                route_key = cond_func(state)
                current = mapping.get(route_key, "END")
                print(f"   🔀 Conditional route: {route_key} → {current}")
            elif current in next_map:
                current = next_map[current]
            else:
                current = "END"

        elapsed = (time.time() - start_time) * 1000
        state["processing_time_ms"] = elapsed

        print("\n" + "=" * 70)
        print(f"✅ WORKFLOW COMPLETED — {len(visited)} nodes executed in {elapsed:.0f}ms")
        print(f"   Path: {' → '.join(visited)}")
        print(f"   Status: {state.get('workflow_status', 'unknown')}")
        print("=" * 70)

        return state


# ── Build the graph ──

def route_by_confidence(state: dict) -> str:
    """Conditional edge: route based on triage confidence."""
    if state.get("triage_error"):
        return "error"
    if state.get("needs_human_review"):
        return "needs_review"
    return "proceed"

# Construct graph
graph = SimpleWorkflowGraph()

# Add nodes
graph.add_node("parse_intake", node_parse_intake)
graph.add_node("triage", node_triage)
graph.add_node("insurance", node_insurance)
graph.add_node("route_schedule", node_route_schedule)
graph.add_node("format_summary", node_format_summary)

# Add edges (linear flow with one conditional)
graph.set_entry_point("parse_intake")
graph.add_edge("parse_intake", "triage")
# Conditional: after triage, check confidence
graph.add_conditional_edges("triage", route_by_confidence, {
    "proceed": "insurance",
    "needs_review": "insurance",  # Still proceed, but flag it
    "error": "format_summary"     # Skip to summary on error
})
graph.add_edge("insurance", "route_schedule")
graph.add_edge("route_schedule", "format_summary")

print("✅ Workflow graph constructed")
print("\nGraph topology:")
print("   parse_intake → triage → [confidence gate] → insurance → route_schedule → format_summary")
print("                                    ↓ (error)")
print("                              format_summary")


In [ ]:
# ============================================================
# Cell 11: Execute the LangGraph Workflow
# ============================================================

# Run the full workflow
final_state = graph.invoke({
    "intake_text": sample_intake
})

# Print the final summary
print("\n" + "=" * 70)
print("📋 FINAL PATIENT SUMMARY")
print("=" * 70)
print(final_state["summary"])
print(f"\n⏱️  Total Processing Time: {final_state['processing_time_ms']:.0f}ms")
if final_state.get("needs_human_review"):
    print("⚠️  FLAGGED FOR HUMAN REVIEW")
if final_state.get("errors"):
    print(f"⚠️  Errors encountered: {final_state['errors']}")


## 3.1 Testing with Different Patient Scenarios

A robust workflow must handle diverse cases. Let's test with a **routine** patient to verify our conditional routing works correctly.


In [ ]:
# ============================================================
# Cell 12: Test with a Routine Patient
# ============================================================

routine_intake = """
Patient: Priya Sharma, 32 year old female

Presenting Complaint: Patient requests prescription refill for her birth control
medication (Levonorgestrel). Last prescription expired 5 days ago.
No new symptoms, no changes in health status.

Past Medical History: None significant
Current Medications: Levonorgestrel 0.15mg daily
Allergies: None known

Vitals: BP 118/72, HR 68, Temp 98.2°F, RR 16, SpO2 99% on room air
"""

# We need to adjust our mock to handle this case
class EnhancedMockClient(MockGeminiClient):
    def generate(self, model, prompt, **kwargs):
        prompt_lower = prompt.lower()

        if "levonorgestrel" in prompt_lower or "birth control" in prompt_lower or "refill" in prompt_lower:
            if "triage" in prompt_lower or "classify" in prompt_lower:
                return MockGeminiResponse(json.dumps({
                    "triage_level": "ROUTINE",
                    "esi_score": 5,
                    "reasoning": "Patient presents for routine prescription refill. No acute "
                                 "complaints, no new symptoms. Vital signs are within normal limits. "
                                 "This is a standard primary care visit requiring no emergency "
                                 "resources. ESI Level 5 — no resources needed.",
                    "confidence": 0.98,
                    "recommended_department": "Primary Care",
                    "time_sensitivity": "Non-urgent — schedule within 3-5 business days",
                    "red_flags": []
                }, indent=2))
            elif "parse" in prompt_lower:
                return MockGeminiResponse(json.dumps({
                    "patient_name": "Priya Sharma",
                    "age": 32,
                    "gender": "Female",
                    "chief_complaint": "Prescription refill for oral contraceptive (Levonorgestrel)",
                    "symptoms": ["prescription expired"],
                    "duration": "5 days since expiry",
                    "medical_history": [],
                    "current_medications": ["Levonorgestrel 0.15mg daily"],
                    "allergies": [],
                    "vital_signs": {
                        "blood_pressure": "118/72",
                        "heart_rate": 68,
                        "temperature": "98.2°F",
                        "respiratory_rate": 16,
                        "oxygen_saturation": "99%"
                    }
                }, indent=2))
            else:
                return super().generate(model, prompt, **kwargs)

        return super().generate(model, prompt, **kwargs)

# Temporarily swap client
original_client = client
client = EnhancedMockClient()

routine_state = graph.invoke({"intake_text": routine_intake})

print("\n" + "=" * 70)
print("📋 ROUTINE PATIENT SUMMARY")
print("=" * 70)
print(routine_state["summary"])

# Restore client
client = original_client


---

# Part 4: Rebuilding with Google ADK

Now let's rebuild the **entire workflow** using Google's Agent Development Kit (ADK). This demonstrates how ADK's built-in workflow agents simplify the implementation.

### Key Differences: LangGraph vs ADK

| Aspect | LangGraph | Google ADK |
|--------|-----------|------------|
| **Graph definition** | Manual StateGraph + edges | Declarative SequentialAgent / ParallelAgent |
| **State management** | TypedDict you manage | Session state managed by ADK |
| **Tool integration** | Wrap in functions | `FunctionTool` decorator |
| **Production deploy** | Self-managed (Cloud Run) | One-click to Agent Engine |
| **Memory** | Custom checkpointer | Built-in Memory Bank |
| **Tracing** | Add Langfuse manually | Built-in tracing |
| **Human-in-the-loop** | Custom interrupt logic | Built-in `escalate()` |

### ADK Architecture for Our Use Case

```
SequentialAgent("patient_intake_pipeline")
├── LlmAgent("intake_parser")         → Gemini Flash + output_key
├── LlmAgent("triage_classifier")     → Gemini Pro + output_key
├── LlmAgent("insurance_checker")     → Gemini Flash + FunctionTool
├── LlmAgent("router_scheduler")      → Gemini Flash + FunctionTool
└── LlmAgent("summary_formatter")     → Gemini Flash + output_key
```


In [ ]:
# ============================================================
# Cell 13: Google ADK — Define Tools
# ============================================================

# In production, you'd import:
# from google.adk.agents import LlmAgent, SequentialAgent
# from google.adk.tools import FunctionTool

# For this notebook, we define the ADK-compatible structure
# that maps directly to the real ADK API.

# ── ADK FunctionTools ──
# In ADK, tools are Python functions decorated with metadata.
# The ADK runtime handles JSON schema generation and call routing.

def check_insurance_eligibility(patient_name: str) -> dict:
    """
    Check patient insurance eligibility against payer systems.

    ADK will auto-generate the JSON schema from this function signature.
    In production:
        @FunctionTool
        def check_insurance_eligibility(patient_name: str) -> dict:
            ...

    Args:
        patient_name: Full name of the patient to verify.

    Returns:
        dict with eligibility status, plan details, and copay information.
    """
    return insurance_api.check_eligibility(patient_name)


def schedule_appointment(
    department: str,
    patient_name: str,
    urgency: str,
    insurance_status: str
) -> dict:
    """
    Schedule a patient appointment in the appropriate department.

    Args:
        department: Target department (e.g., 'Emergency Department', 'Primary Care')
        patient_name: Full name of the patient
        urgency: Triage urgency level (EMERGENCY, URGENT, LESS_URGENT, ROUTINE)
        insurance_status: Insurance eligibility status

    Returns:
        dict with appointment details and confirmation.
    """
    # Map urgency to routing
    route = ROUTING_TABLE.get(TriageLevel(urgency), ROUTING_TABLE[TriageLevel.ROUTINE])

    return {
        "appointment_scheduled": True,
        "department": department,
        "arrival_instruction": route["action"],
        "estimated_wait": route["estimated_wait"],
        "notification_sent": True,
        "notification_channels": ["SMS", "Patient Portal"],
        "confirmation_id": f"APT-{uuid.uuid4().hex[:8].upper()}"
    }


def send_notification(
    patient_name: str,
    message: str,
    channels: list[str]
) -> dict:
    """
    Send notification to patient via specified channels.

    Args:
        patient_name: Patient to notify
        message: Notification message text
        channels: List of channels (SMS, Email, Patient Portal)

    Returns:
        dict with notification delivery status.
    """
    return {
        "sent": True,
        "patient_name": patient_name,
        "channels": channels,
        "timestamp": datetime.now().isoformat()
    }


print("✅ ADK FunctionTools defined:")
print("   • check_insurance_eligibility(patient_name) → dict")
print("   • schedule_appointment(department, patient_name, urgency, insurance_status) → dict")
print("   • send_notification(patient_name, message, channels) → dict")


In [ ]:
# ============================================================
# Cell 14: Google ADK — Production Pipeline Code
# ============================================================
# This cell shows the COMPLETE ADK implementation.
# Run this directly in your environment with: pip install google-adk

ADK_CODE = """
# ── Production ADK Implementation ──
# pip install google-adk

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import FunctionTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# Step 1: Intake Parser Agent
intake_parser = LlmAgent(
    name="intake_parser",
    model="gemini-2.5-flash",
    instruction=(
        "You are a medical intake specialist. Parse the patient intake "
        "information and extract structured clinical data. "
        "Extract: name, age, gender, chief complaint, symptoms, duration, "
        "medical history, medications, allergies, vital signs. "
        "Normalize medication names to generic + dosage."
    ),
    output_key="parsed_intake"
)

# Step 2: Triage Classifier Agent (Gemini Pro for clinical reasoning)
triage_classifier = LlmAgent(
    name="triage_classifier",
    model="gemini-2.5-pro",
    instruction=(
        "You are an experienced triage nurse using ESI v4. "
        "Based on the parsed intake data: {parsed_intake} "
        "Classify: EMERGENCY (ESI 1), URGENT (ESI 2), "
        "LESS_URGENT (ESI 3), or ROUTINE (ESI 4-5). "
        "CRITICAL: When in doubt, choose MORE URGENT. "
        "Chest pain + age >40 + cardiac risk = minimum ESI 2. "
        "Provide: triage_level, esi_score, reasoning, confidence, "
        "recommended_department, time_sensitivity, red_flags."
    ),
    output_key="triage_result"
)

# Step 3: Insurance Checker (Flash + Tool)
insurance_checker = LlmAgent(
    name="insurance_checker",
    model="gemini-2.5-flash",
    instruction=(
        "Check insurance eligibility for the patient from {parsed_intake}. "
        "Use the check_insurance_eligibility tool. "
        "Report plan, eligibility, copay, and network status."
    ),
    tools=[FunctionTool(check_insurance_eligibility)],
    output_key="insurance_result"
)

# Step 4: Router & Scheduler (Flash + Tools)
router_scheduler = LlmAgent(
    name="router_scheduler",
    model="gemini-2.5-flash",
    instruction=(
        "Based on triage: {triage_result} and insurance: {insurance_result}, "
        "schedule an appointment using schedule_appointment tool, "
        "then notify via send_notification tool."
    ),
    tools=[
        FunctionTool(schedule_appointment),
        FunctionTool(send_notification)
    ],
    output_key="scheduling_result"
)

# Step 5: Summary Formatter
summary_formatter = LlmAgent(
    name="summary_formatter",
    model="gemini-2.5-flash",
    instruction=(
        "Generate a clinical summary combining: "
        "{parsed_intake}, {triage_result}, {insurance_result}, {scheduling_result}. "
        "Format as a professional clinical note."
    ),
    output_key="final_summary"
)

# Compose into Sequential Workflow
patient_intake_pipeline = SequentialAgent(
    name="patient_intake_pipeline",
    sub_agents=[
        intake_parser, triage_classifier, insurance_checker,
        router_scheduler, summary_formatter
    ],
    description="End-to-end patient intake, triage, and scheduling"
)

# Run
session_service = InMemorySessionService()
runner = Runner(
    agent=patient_intake_pipeline,
    app_name="patient_intake",
    session_service=session_service
)
session = session_service.create_session(
    app_name="patient_intake", user_id="staff-001"
)
response = runner.run(
    session_id=session.id,
    user_id="staff-001",
    new_message=(
        "Patient: Rajesh Kumar, 58M. Severe chest pain radiating to "
        "left arm, 2hr, SOB, diaphoresis. Hx: DM2, HTN, HLD. "
        "Meds: Metformin, Amlodipine, Atorvastatin. Allergy: PCN. "
        "VS: BP 165/95, HR 102, T 98.6, RR 22, SpO2 94 pct"
    )
)
for event in response:
    if event.text:
        print(event.text)
"""

print("ADK Pipeline Code (ready to run with google-adk installed)")
print("=" * 70)
print(ADK_CODE)


## 4.1 Simulated ADK Execution

Since the ADK requires API credentials, let's simulate the execution to show the expected behavior.


In [ ]:
# ============================================================
# Cell 15: Simulated ADK Execution
# ============================================================

class ADKSimulator:
    """Simulates ADK SequentialAgent execution for demonstration."""

    def run_pipeline(self, intake_text: str):
        run_id = uuid.uuid4().hex[:8]
        print("=" * 70)
        print(f"🚀 ADK SequentialAgent Pipeline — Run: {run_id}")
        print("=" * 70)

        session_state = {}
        total_start = time.time()

        # ── Step 1: intake_parser (Gemini Flash) ──
        print("\n━━━ Step 1/5: intake_parser (gemini-2.5-flash) ━━━")
        start = time.time()
        response = client.generate(model="gemini-2.5-flash",
                                   prompt=f"Parse intake: {intake_text}")
        parsed = json.loads(response.text)
        patient = PatientIntake(**parsed)
        session_state["parsed_intake"] = parsed
        print(f"   👤 {patient.patient_name}, {patient.age}{patient.gender[0]} "
              f"— {len(patient.symptoms)} symptoms")
        print(f"   ⏱️  {(time.time()-start)*1000:.0f}ms")

        # ── Step 2: triage_classifier (Gemini Pro) ──
        print("\n━━━ Step 2/5: triage_classifier (gemini-2.5-pro) ━━━")
        start = time.time()
        triage, needs_review = classify_triage(patient)
        session_state["triage_result"] = triage.model_dump()
        emoji = {"EMERGENCY":"🔴","URGENT":"🟠","LESS_URGENT":"🟡","ROUTINE":"🟢"}
        print(f"   {emoji.get(triage.triage_level,'⚪')} {triage.triage_level} "
              f"(ESI {triage.esi_score}, confidence: {triage.confidence:.0%})")
        print(f"   ⏱️  {(time.time()-start)*1000:.0f}ms")
        if needs_review:
            print("   ⚠️  Flagged for human review (low confidence)")

        # ── Step 3: insurance_checker (Flash + Tool) ──
        print("\n━━━ Step 3/5: insurance_checker (gemini-2.5-flash + tool) ━━━")
        start = time.time()
        print(f"   🔧 Tool call: check_insurance_eligibility('{patient.patient_name}')")
        ins = check_insurance(patient.patient_name)
        session_state["insurance_result"] = ins.model_dump()
        print(f"   💳 {ins.plan} — {'✅ Eligible' if ins.eligible else '❌ Not Eligible'}")
        print(f"   ⏱️  {(time.time()-start)*1000:.0f}ms")

        # ── Step 4: router_scheduler (Flash + Tools) ──
        print("\n━━━ Step 4/5: router_scheduler (gemini-2.5-flash + tools) ━━━")
        start = time.time()
        sched = route_and_schedule(triage, patient, ins)
        session_state["scheduling_result"] = sched.model_dump()
        conf_id = f"APT-{uuid.uuid4().hex[:8].upper()}"
        print(f"   🔧 Tool call: schedule_appointment(dept='{sched.department}', ...)")
        print(f"   🔧 Tool call: send_notification(channels={sched.notification_channels})")
        print(f"   📋 Confirmation: {conf_id}")
        print(f"   ⏱️  {(time.time()-start)*1000:.0f}ms")

        # ── Step 5: summary_formatter (Flash) ──
        print("\n━━━ Step 5/5: summary_formatter (gemini-2.5-flash) ━━━")
        start = time.time()
        response = client.generate(model="gemini-2.5-flash",
                                   prompt=f"Format summary for {patient.patient_name}")
        session_state["final_summary"] = response.text
        print(f"   ✅ Summary generated")
        print(f"   ⏱️  {(time.time()-start)*1000:.0f}ms")

        total_ms = (time.time() - total_start) * 1000

        print("\n" + "=" * 70)
        print(f"✅ ADK Pipeline Complete — {total_ms:.0f}ms total")
        print("=" * 70)

        print("\n📋 FINAL CLINICAL NOTE:")
        print("-" * 50)
        print(response.text.replace("\\n", "\n"))

        return session_state


# Run it
simulator = ADKSimulator()
adk_result = simulator.run_pipeline(sample_intake)


---

# Part 5: Comparison & Production Considerations

## 5.1 LangGraph vs ADK — Side by Side

| Dimension | LangGraph Approach | ADK Approach |
|-----------|-------------------|--------------|
| **Lines of code** | ~120 (graph + nodes + edges) | ~80 (agents + compose) |
| **State management** | Manual TypedDict + merge | Automatic session state |
| **Conditional routing** | `add_conditional_edges()` | Agent instructions + `escalate()` |
| **Tool integration** | Wrap in node functions | `FunctionTool` decorator |
| **Error handling** | Custom per-node try/except | Built-in with retry policies |
| **Testing** | Test each node independently | Test each sub-agent |
| **Deployment** | Cloud Run + custom infra | `Agent Engine` one-click deploy |
| **When to choose** | Max flexibility, complex graphs | Faster dev, GCP-native, production speed |

## 5.2 Production Checklist

### Security (HIPAA Compliance)
- [ ] VPC Service Controls perimeter around all GCP services
- [ ] Customer-Managed Encryption Keys (CMEK) for all data at rest
- [ ] PHI redaction in all logs (Cloud Logging + Langfuse)
- [ ] Business Associate Agreement (BAA) with GCP
- [ ] Audit trail for every patient data access
- [ ] Service account least-privilege IAM

### Reliability
- [ ] Retry with exponential backoff on all external API calls
- [ ] Circuit breaker pattern for insurance API
- [ ] Fallback responses when LLM is unavailable
- [ ] Maximum latency SLO: p99 < 5s for routine, < 2s for emergency
- [ ] Dead letter queue for failed processing

### Observability
- [ ] Cloud Trace spans for every node execution
- [ ] Langfuse traces for LLM calls (prompt, response, tokens, cost)
- [ ] Dashboard: triage distribution, processing time, error rate
- [ ] Alert: triage confidence < 0.5 more than 3 times in 1 hour
- [ ] Weekly eval: compare agent triage vs nurse triage on 100 cases

### Cost Optimization
- [ ] Gemini Flash for all steps except triage (Pro for clinical reasoning)
- [ ] Context caching for triage prompt (same system instructions across calls)
- [ ] Batch processing for non-urgent intakes (off-peak hours)
- [ ] Model routing: Flash for ESI 4-5, Pro for ESI 1-3


In [ ]:
# ============================================================
# Cell 16: Production Deployment Architecture
# ============================================================

deployment_config = {
    "service": "patient-intake-agent",
    "platform": "Vertex AI Agent Engine",
    "runtime": {
        "agent_framework": "Google ADK",
        "agent_type": "SequentialAgent",
        "sub_agents": 5,
    },
    "models": {
        "intake_parser": "gemini-2.5-flash",
        "triage_classifier": "gemini-2.5-pro",
        "insurance_checker": "gemini-2.5-flash",
        "router_scheduler": "gemini-2.5-flash",
        "summary_formatter": "gemini-2.5-flash"
    },
    "tools": [
        "check_insurance_eligibility (Availity API)",
        "schedule_appointment (Epic FHIR R4)",
        "send_notification (Twilio SMS + SendGrid Email)"
    ],
    "infrastructure": {
        "api_gateway": "Cloud Run (FastAPI)",
        "database": "AlloyDB PostgreSQL (patient records)",
        "document_store": "Cloud Storage (intake forms, PDFs)",
        "vector_db": "Vertex AI Vector Search (medical KB)",
        "events": "Pub/Sub (async notifications)",
        "cache": "Memorystore Redis (session state)",
    },
    "security": {
        "network": "VPC-SC perimeter",
        "encryption": "CMEK (Cloud KMS)",
        "auth": "IAM + Workload Identity Federation",
        "compliance": "HIPAA BAA with GCP",
        "data_residency": "us-central1 (single region)"
    },
    "observability": {
        "tracing": "Cloud Trace + Langfuse",
        "logging": "Cloud Logging (PHI-redacted)",
        "monitoring": "Cloud Monitoring + custom dashboards",
        "alerting": "PagerDuty integration via Cloud Monitoring"
    },
    "slos": {
        "availability": "99.9%",
        "latency_p99_emergency": "< 2 seconds",
        "latency_p99_routine": "< 5 seconds",
        "triage_accuracy": "> 95%",
        "error_rate": "< 0.1%"
    }
}

print("🏗️  PRODUCTION DEPLOYMENT CONFIGURATION")
print("=" * 60)

for section, config in deployment_config.items():
    print(f"\n📦 {section.upper().replace('_', ' ')}")
    if isinstance(config, dict):
        for k, v in config.items():
            print(f"   {k}: {v}")
    elif isinstance(config, list):
        for item in config:
            print(f"   • {item}")
    else:
        print(f"   {config}")


---

# Summary

In this notebook, we built a **production-grade Patient Intake & Clinical Triage Agent** that:

### What We Built
1. **Parsed** unstructured intake forms into structured clinical data using Gemini Flash
2. **Classified** triage urgency using Gemini Pro with ESI v4 guidelines and confidence gating
3. **Verified** insurance eligibility with retry logic and graceful fallbacks
4. **Routed** patients to departments using deterministic rules (not LLM — more auditable)
5. **Scheduled** appointments and sent notifications via tool integrations

### Two Implementation Approaches
- **LangGraph**: Maximum control, explicit graph construction, custom state management
- **Google ADK**: Declarative agents, built-in session state, one-click deploy to Agent Engine

### Key Architecture Decisions
- **Gemini Flash** for fast, simple steps (parsing, insurance, scheduling)
- **Gemini Pro** for the critical step (triage — where accuracy saves lives)
- **Deterministic routing** (not LLM) for department assignment — predictable and auditable
- **Confidence gating** at 80% — flag low-confidence triage for nurse review
- **Pydantic validation** at every step — catch LLM errors before they hit downstream systems
- **Retry + fallback** on external APIs — never block emergency care for an API timeout

### Next Steps
1. Connect to real EHR system (Epic FHIR R4 / Cerner)
2. Add Document AI for PDF/image intake form processing
3. Implement LoopAgent for triage quality review (write → review → revise)
4. Deploy to Vertex AI Agent Engine with Cloud Trace observability
5. Run eval pipeline: 500 historical cases, compare agent vs nurse triage

---
*Generative AI Architect Program — Batch 7 | Blue Data Consulting | April 2026*
